In [1]:
import sys
import os
import torch
from pathlib import Path

def get_project_info() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    root = current
    for parent in [current, *current.parents]:
        if (parent / "toy_transformers").exists():
            root = parent
            break
    return root, current

if 'ROOT_DIR' not in globals():
    ROOT_DIR, EXPERIMENT_DIR = get_project_info()
    if str(ROOT_DIR) not in sys.path:
        sys.path.append(str(ROOT_DIR))
    if Path.cwd() != ROOT_DIR:
        os.chdir(ROOT_DIR)

from toy_transformers.models import gptv4
from toy_transformers import tokenization
from toy_transformers.model_io import load_model
from toy_transformers.data import S3Sync
from toy_transformers.config import TrainingConfig

print(f"ROOT_DIR: {ROOT_DIR}")

ROOT_DIR: /Users/sselva890@cable.comcast.com/dev/apps/toy-transformers


In [2]:
RUN_NAME = "baseline-fineweb-edu"
CHECKPOINT = "final"
DEVICE = "mps"  # change to 'cuda' or 'cpu' as needed

cfg = TrainingConfig.from_json(ROOT_DIR / "configs" / f"{RUN_NAME}.json")
cfg.tokenizer.load(ROOT_DIR)
print(f"Vocab size: {cfg.tokenizer.vocab_size}, BOS={cfg.tokenizer.bos_id}, PAD={cfg.tokenizer.pad_id}")

CKPT_DIR = ROOT_DIR / f"runs/{RUN_NAME}/checkpoints/{CHECKPOINT}"

torch.set_float32_matmul_precision("medium")

model = cfg.model.build_model(vocab_size=cfg.tokenizer.vocab_size, device=DEVICE)
print(f"{model.get_num_parameters(as_str=True)} parameters")

load_model(CKPT_DIR, cfg, model, device=DEVICE)
model.eval()
print("Checkpoint loaded successfully")

vocab = tokenization.Vocabulary.load(ROOT_DIR / cfg.tokenizer.path)

def encode(text: str) -> torch.Tensor:
    ids = [cfg.tokenizer.bos_id] + vocab.encode(text.encode("utf-8"))
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)

def decode_tokens(token_ids: list[int]) -> str:
    byte_chunks = vocab.decode(token_ids)
    return b"".join(byte_chunks).decode("utf-8", errors="replace")

print("vocab + model loaded")

Vocab size: 32768, BOS=0, PAD=1
100.682m parameters
Checkpoint loaded successfully
vocab + model loaded


In [3]:
for name, p in model.named_parameters():
	print(f"{name}: {p.norm().item()} | {p.var()} | {p.shape}")

token_embed.weight: 163.0474853515625 | 0.0010563719552010298 | torch.Size([32768, 768])
blocks.0.norm1.weight: 26.97032356262207 | 0.0019365240586921573 | torch.Size([768])
blocks.0.attn.q_proj.weight: 20.948762893676758 | 0.0007440369226969779 | torch.Size([768, 768])
blocks.0.attn.kv_proj.weight: 15.138236045837402 | 0.0005828012363053858 | torch.Size([512, 768])
blocks.0.attn.proj.weight: 16.227262496948242 | 0.0004464449593797326 | torch.Size([768, 768])
blocks.0.norm2.weight: 20.349044799804688 | 0.002735547022894025 | torch.Size([768])
blocks.0.mlp.l1.weight: 38.338016510009766 | 0.0006229770369827747 | torch.Size([3072, 768])
blocks.0.mlp.proj.weight: 32.222862243652344 | 0.00044009406701661646 | torch.Size([768, 3072])
blocks.1.norm1.weight: 25.244258880615234 | 0.0013862868072465062 | torch.Size([768])
blocks.1.attn.q_proj.weight: 20.82854461669922 | 0.0007355217239819467 | torch.Size([768, 768])
blocks.1.attn.kv_proj.weight: 14.919854164123535 | 0.0005661070463247597 | torch

In [4]:
import torch
import torch.nn.functional as F
from collections import defaultdict
import math

attention_entropies = defaultdict(list)

def patched_attn_forward(self, x, block_mask=None):
    B, T, _ = x.size()

    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    kv = self.kv_proj(x).view(B, T, self.n_kv_heads, 2 * self.head_dim)
    k, v = kv.split(self.head_dim, dim=-1)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)

    q, k = self.rope(q, k)

    # expand KV heads to match Q heads (GQA)
    n_rep = self.n_heads // self.n_kv_heads
    k = k.repeat_interleave(n_rep, dim=1)  # (B, n_heads, T, head_dim)
    v = v.repeat_interleave(n_rep, dim=1)

    # manual attention to expose weights
    scale = 1.0 / math.sqrt(self.head_dim)
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, H, T, T)
    # causal mask
    causal_mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
    scores = scores.masked_fill(~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
    attn_weights = F.softmax(scores.float(), dim=-1)  # (B, H, T, T)

    # store entropy
    entropy = -(attn_weights * (attn_weights + 1e-9).log()).sum(dim=-1)  # (B, H, T)
    self._last_entropy = entropy.detach().cpu()

    y = torch.matmul(attn_weights.to(v.dtype), v)
    y = y.transpose(1, 2).contiguous().view(B, T, self.n_embed)
    return self.proj(y)

# patch and run
from types import MethodType
original_forwards = {}
for i, block in enumerate(model.blocks):
    original_forwards[i] = block.attn.forward
    block.attn.forward = MethodType(patched_attn_forward, block.attn)

# --- generation ---
prompt = "<BOS>The best way to heat up a pizza"
MAX_NEW_TOKENS = 100
TEMPERATURE = 1.0
TOPK = 40

idx = encode(prompt)
generated_tokens = []
print(prompt, end="", flush=True)

with torch.no_grad():
    for token in model.generate(idx, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, topk=TOPK):
        if token.item() == cfg.tokenizer.bos_id:
            break
        tok = token.item()
        generated_tokens.append(tok)
        print(decode_tokens([tok]), end="", flush=True)
        # collect entropy from this generation step
        for i, block in enumerate(model.blocks):
            if hasattr(block.attn, '_last_entropy'):
                attention_entropies[i].append(block.attn._last_entropy)

print()

# restore original forwards
for i, block in enumerate(model.blocks):
    block.attn.forward = original_forwards[i]

# --- summary ---
print("\n--- attention entropy per layer ---")
for layer_idx in sorted(attention_entropies.keys()):
    # each step: (B, H, T) where T grows — take the last token's entropy only
    last_token_entropies = torch.stack([
        step[:, :, -1] for step in attention_entropies[layer_idx]
    ], dim=-1)  # (B, H, steps)
    mean = last_token_entropies.mean().item()
    per_head = last_token_entropies.mean(dim=-1).squeeze(0)  # (H,)
    head_str = " ".join(f"{v:.2f}" for v in per_head.tolist())
    print(f"block {layer_idx:2d}: mean={mean:.3f} | heads: [{head_str}]")

<BOS>The best way to heat up a pizza is to eat them fresh. You will quickly feel the energy expended and you'll probably feel a bit nauseous, so it's important to eat fresh. However, it's usually not feasible to purchase fresh-picked pizza to get a full serving size, because the nutrients are often difficult to come by and may not be easy to cook on the stovetop. To make sure you get a nutritious slice without frying it, make sure to use the right amount of seasoning and cooking method

--- attention entropy per layer ---
block  0: mean=2.737 | heads: [1.83 3.14 1.96 3.17 2.56 2.72 3.09 2.94 3.04 2.87 3.12 2.41]
block  1: mean=2.578 | heads: [2.24 2.22 2.29 1.94 1.25 2.33 3.20 3.14 3.05 3.04 3.37 2.87]
block  2: mean=2.533 | heads: [2.83 2.32 2.25 2.13 2.80 2.90 2.07 1.77 1.71 3.04 3.22 3.37]
block  3: mean=2.712 | heads: [2.96 3.10 3.00 2.42 1.92 2.15 2.47 2.66 2.70 2.52 3.31 3.34]
block  4: mean=2.504 | heads: [2.15 2.05 2.37 1.99 2.34 2.82 2.86 3.08 2.14 2.89 2.68 2.66]
block  5: me

In [5]:
import torch
from collections import defaultdict
from pathlib import Path
from torch.utils.data import DataLoader
from toy_transformers.data import ShardDataset

# --- config ---
SHARD_PATH = Path("data/datasets/fineweb-edu-10b/shard_0077.bin")
N_SAMPLES = 512
BATCH_SIZE = 8
DEVICE = next(model.parameters()).device

# --- dataset ---
ds = ShardDataset(
    shard_paths=[SHARD_PATH],
    block_size=cfg.model.config["block_size"],
    bos_id=cfg.tokenizer.bos_id,
    pad_id=cfg.tokenizer.pad_id,
    shuffle=False,
)
loader = DataLoader(ds, batch_size=BATCH_SIZE)

# --- hooks ---
dead_accumulator = defaultdict(list)
residual_accumulator = defaultdict(list)
residual_inputs = {}
neuron_ever_active = {}

def make_mlp_hook(layer_idx):
    def hook(module, input, output):
        # per-batch mean sparsity
        dead = (output <= 0).float().mean().item()
        dead_accumulator[layer_idx].append(dead)
        # per-neuron: has this neuron ever fired across all tokens in this batch?
        active = (output > 0).any(dim=(0, 1))  # (4*n_embed,)
        if layer_idx not in neuron_ever_active:
            neuron_ever_active[layer_idx] = active.cpu()
        else:
            neuron_ever_active[layer_idx] |= active.cpu()
    return hook

def make_block_pre_hook(layer_idx):
    def hook(module, input):
        residual_inputs[layer_idx] = input[0].detach()
    return hook

def make_block_post_hook(layer_idx):
    def hook(module, input, output):
        x_in = residual_inputs[layer_idx]
        delta = output - x_in
        ratio = delta.norm(dim=-1) / (x_in.norm(dim=-1) + 1e-8)
        residual_accumulator[layer_idx].append(ratio.mean().item())
    return hook

hooks = []
for i, block in enumerate(model.blocks):
    hooks.append(block.mlp.l1.register_forward_hook(make_mlp_hook(i)))
    hooks.append(block.register_forward_pre_hook(make_block_pre_hook(i)))
    hooks.append(block.register_forward_hook(make_block_post_hook(i)))

# --- forward passes ---
model.eval()
n_seen = 0
with torch.no_grad():
    for batch in loader:
        print('.', end="")
        x, y, doc_ids, loss_mask = batch
        x = x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            model(x, eval_logits=True)
        n_seen += x.size(0)
        if n_seen >= N_SAMPLES:
            break
print(flush=True)
for h in hooks:
    h.remove()

# --- results ---
print(f"\n--- MLP dead neuron fraction (ReLU²) [{n_seen} samples] ---")
for i in sorted(dead_accumulator):
    mean = sum(dead_accumulator[i]) / len(dead_accumulator[i])
    bar = "█" * int(mean * 40)
    print(f"block {i:2d}: {mean:.3f} {bar}")

print(f"\n--- residual stream update ratio (||delta|| / ||x||) [{n_seen} samples] ---")
for i in sorted(residual_accumulator):
    mean = sum(residual_accumulator[i]) / len(residual_accumulator[i])
    bar = "█" * int(mean * 40)
    print(f"block {i:2d}: {mean:.3f} {bar}")

print(f"\n--- permanently dead neurons (never activated over {n_seen} samples) ---")
for i in sorted(neuron_ever_active):
    dead_frac = (~neuron_ever_active[i]).float().mean().item()
    bar = "█" * int(dead_frac * 40)
    print(f"block {i:2d}: {dead_frac:.3f} {bar}")

................................................................

--- MLP dead neuron fraction (ReLU²) [512 samples] ---
block  0: 0.803 ████████████████████████████████
block  1: 0.832 █████████████████████████████████
block  2: 0.863 ██████████████████████████████████
block  3: 0.838 █████████████████████████████████
block  4: 0.812 ████████████████████████████████
block  5: 0.806 ████████████████████████████████
block  6: 0.782 ███████████████████████████████
block  7: 0.794 ███████████████████████████████
block  8: 0.743 █████████████████████████████
block  9: 0.705 ████████████████████████████
block 10: 0.671 ██████████████████████████
block 11: 0.631 █████████████████████████

--- residual stream update ratio (||delta|| / ||x||) [512 samples] ---
block  0: 0.756 ██████████████████████████████
block  1: 0.434 █████████████████
block  2: 0.376 ███████████████
block  3: 0.363 ██████████████
block  4: 0.392 ███████████████
block  5: 0.417 ████████████████
block  6: 0.502 ████████████